# 07 - Can Machine Learning Predict the Stock Market? 📉📈

Let's try to predict Apple's stock price with what we just learned, and discover three of the most important lessons in data science:

1. **Always compare against a simple baseline.**
2. **With time-series data, never let the future leak into training.**
3. **A high score doesn't mean the model learned something useful.**

> ⚠️ Not financial advice! This notebook is about *evaluating models honestly*.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)   # hides harmless math warnings some Macs print

aapl = pd.read_csv("data/AAPL.csv", parse_dates=["Date"], index_col="Date")
df = aapl[["Close"]].copy()
df["Tomorrow"] = df["Close"].shift(-1)       # tomorrow's close, lined up with today's row
df = df.dropna()
df.tail()

## Attempt 1: predict tomorrow's price from today's price

**Split by time, not randomly.** We train on the first 80% of days and test on the last 20%. A random shuffle would let the model train on days *after* the test days. That's "peeking at the future," which is impossible in real life (this is called **data leakage**).

In [ ]:
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
print("train:", train.index.min().date(), "to", train.index.max().date())
print("test: ", test.index.min().date(), "to", test.index.max().date())

model = LinearRegression()
model.fit(train[["Close"]], train["Tomorrow"])
pred = model.predict(test[["Close"]])

print(f"\nR^2 on test: {r2_score(test['Tomorrow'], pred):.3f}")
print(f"Average error: ${mean_absolute_error(test['Tomorrow'], pred):.2f}")

An R² near 1 looks amazing! 🎉 ...But wait. Let's compare with the **dumbest possible strategy**: *"tomorrow's price = today's price."*

In [ ]:
baseline = test["Close"]                    # predict "no change"
print(f"Baseline R^2:         {r2_score(test['Tomorrow'], baseline):.3f}")
print(f"Baseline avg error:   ${mean_absolute_error(test['Tomorrow'], baseline):.2f}")
print(f"Model avg error:      ${mean_absolute_error(test['Tomorrow'], pred):.2f}")
print(f"Model: tomorrow = {model.coef_[0]:.4f} * today + {model.intercept_:.2f}")

The model basically learned **"tomorrow ≈ today"**: its slope is about 1. The impressive R² comes from the fact that prices don't jump far in one day, not from any real insight. **Lesson 1: a model is only good if it beats a simple baseline.**

In [ ]:
last = test.iloc[-60:]
plt.figure(figsize=(12, 4))
plt.plot(last.index, last["Tomorrow"], label="actual")
plt.plot(last.index, model.predict(last[["Close"]]), label="model", linestyle="--")
plt.title("The model just follows one day behind")
plt.legend()
plt.show()

## Attempt 2: predict tomorrow's *change* (the part that actually matters)
If you want to make money, you need to know whether tomorrow goes **up or down**. Let's use the last 5 days of returns as features.

In [ ]:
r = pd.DataFrame({"ret": aapl["Close"].pct_change()})
for lag in range(1, 6):
    r[f"lag{lag}"] = r["ret"].shift(lag)       # the return 1, 2, ..., 5 days ago
r["target"] = r["ret"].shift(-1)               # tomorrow's return
r = r.dropna()
LAGS = [f"lag{i}" for i in range(1, 6)]

split = int(len(r) * 0.8)
tr, te = r.iloc[:split], r.iloc[split:]
reg = LinearRegression().fit(tr[LAGS], tr["target"])
print(f"R^2 predicting tomorrow's return: {r2_score(te['target'], reg.predict(te[LAGS])):.4f}")

An R² around 0, or even **below** 0, means the model is no better (or worse!) than always guessing the average return.

What about just predicting **up vs down** (classification)?

In [ ]:
tr_up = (tr["target"] > 0).astype(int)
te_up = (te["target"] > 0).astype(int)
clf = LogisticRegression().fit(tr[LAGS], tr_up)
print(f"Model accuracy (up/down): {clf.score(te[LAGS], te_up):.1%}")
print(f"Baseline 'always guess UP': {te_up.mean():.1%}")

About a coin flip, and no better than always guessing "up" (stocks rise slightly more often than they fall).

## Attempt 3: fit a trend line to the date (a common mistake)
A tempting idea is to use the **day number** as the only feature, i.e. fit a trend line through time. Look what happens when a straight line fitted to the past is used to predict the future:

In [ ]:
days = np.arange(len(aapl)).reshape(-1, 1)
split = int(len(aapl) * 0.8)
trend = LinearRegression().fit(days[:split], aapl["Close"].iloc[:split])

plt.figure(figsize=(12, 4))
plt.plot(aapl.index, aapl["Close"], label="actual")
plt.plot(aapl.index, trend.predict(days), label="trend line", linestyle="--")
plt.axvline(aapl.index[split], color="gray", linestyle=":", label="train | test")
plt.legend()
plt.title("A trend line fitted to the past is just a guess about the future")
plt.show()
print(f"Test R^2 of the trend line: {trend.score(days[split:], aapl['Close'].iloc[split:]):.2f}")

## Why is this so hard?
- **Efficient market hypothesis:** if a pattern were easy to find, traders would already have used it, and using it makes it disappear.
- Prices react to **news** (earnings, tariffs, a new iPhone...) that isn't in yesterday's prices.
- There's lots of **noise**: most daily moves are essentially random.

## The three lessons (they apply to ALL of machine learning)
1. **Compare against a baseline.** "Accuracy 53%" means nothing until you know that "always guess up" gets 53%.
2. **No leakage.** Split time-series data by time. Fit scalers and other preprocessing on the training data only.
3. **Be skeptical of great scores.** Ask what the model *actually* learned.

## ✏️ Try it
1. Repeat Attempt 1 with NVDA or KO. Does the model ever beat the baseline?
2. Add the 20-day moving average as a feature in Attempt 2. Any better?
3. What happens in Attempt 1 if you use a *random* split (`train_test_split(..., shuffle=True)`)? Why is that result misleading?

In [ ]:
# your code here